# Logarithms, Exponents & the Log Scale — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/foundations/1-2/logarithms-and-exponents.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/foundations/1-2/logarithms-and-exponents.ipynb)

> Companion to the **Logarithms, Exponents & the Log Scale** article. Watch three ideas
> run, line by line: a **log turns multiplication into addition**, the **log scale** makes
> `0.5` and `2` equidistant from `1`, and the **logit ↔ sigmoid** pair round-trips a
> probability. Pure `numpy`/`math` — **no API key, nothing to install** on Colab.

## What you'll build

1. A one-line proof that **`log(a·b) = log a + log b`** (and the ratio version).
2. A printed **log axis** showing that `0.5` (÷2) and `2` (×2) sit the **same distance** from `1`.
3. The **logit → sigmoid** round trip: push a probability out to log-odds and back, exactly.

**No API key needed.** Everything is deterministic `numpy` arithmetic, so the notebook runs
top-to-bottom anywhere. Each section prints what it computes so you can *see* the property hold.

### Setup (numpy is already available on Colab)

In [1]:
import numpy as np
print("ready — pure numpy, no API key required")

ready — pure numpy, no API key required


## 1. The key property — log turns × into +

The one fact that makes logs indispensable: multiplication inside a log becomes **addition**,
and division becomes **subtraction**. We check both on `a = 8`, `b = 4`.

In [2]:
a, b = 8.0, 4.0

lhs = np.log(a * b)             # log of the PRODUCT
rhs = np.log(a) + np.log(b)     # SUM of the logs
print(f"log(a*b)        = {lhs:.6f}")
print(f"log a + log b   = {rhs:.6f}")
print(f"equal? {np.isclose(lhs, rhs)}")

# a ratio becomes a difference
print(f"\nlog(a/b)        = {np.log(a / b):.6f}")
print(f"log a - log b   = {np.log(a) - np.log(b):.6f}")
print(f"equal? {np.isclose(np.log(a / b), np.log(a) - np.log(b))}")

# base-2 view of the same example from the article: log2(8*4) = 3 + 2 = 5
log2 = lambda x: np.log(x) / np.log(2)
print(f"\nlog2(8*4) = {log2(8*4):.1f}   log2(8) + log2(4) = {log2(8):.1f} + {log2(4):.1f}")

log(a*b)        = 3.465736
log a + log b   = 3.465736
equal? True

log(a/b)        = 0.693147
log a - log b   = 0.693147
equal? True

log2(8*4) = 5.0   log2(8) + log2(4) = 3.0 + 2.0


## 2. The log scale — `0.5` and `2` are equidistant from `1`

On a **log** axis, equal *ratios* are equal *distances*. Halving (÷2 → `0.5`) and doubling
(×2 → `2`) are mirror moves, so they land the same distance either side of the `1` no-effect
line. The printer below places each value by its natural log.

In [3]:
def log_axis(values, width=44):
    """Print values on a natural-log axis so equal RATIOS are equal DISTANCES."""
    logs = np.log(values)
    lo, hi = logs.min(), logs.max()
    print("log axis (equal ratios = equal gaps):")
    for v, lg in zip(values, logs):
        col = int((lg - lo) / (hi - lo) * (width - 1))
        line = [" "] * width
        line[col] = "|"
        marker = "  <- no-effect (ratio = 1)" if np.isclose(v, 1.0) else ""
        print(f"  {v:>4} {''.join(line)}{marker}")

log_axis(np.array([0.25, 0.5, 1.0, 2.0, 4.0]))

d_half = abs(np.log(0.5) - np.log(1.0))
d_two  = abs(np.log(2.0) - np.log(1.0))
print(f"\ndistance(0.5 -> 1) = {d_half:.4f}")
print(f"distance(2   -> 1) = {d_two:.4f}")
print(f"equal distance? {np.isclose(d_half, d_two)}")
print("this symmetry is WHY ratio confidence intervals (RR, OR) live on the log scale")

log axis (equal ratios = equal gaps):
  0.25 |                                           
   0.5           |                                 
   1.0                      |                        <- no-effect (ratio = 1)
   2.0                                 |           
   4.0                                            |

distance(0.5 -> 1) = 0.6931
distance(2   -> 1) = 0.6931
equal distance? True
this symmetry is WHY ratio confidence intervals (RR, OR) live on the log scale


## 3. The logit ↔ sigmoid round trip

The **logit** stretches a probability from `(0,1)` onto the whole real line as a **log-odds**.
The **sigmoid** is its exact inverse and squashes it back. This pair is the skeleton of
logistic regression: predict a log-odds, then sigmoid it into a probability.

In [4]:
def logit(p):
    """Probability -> log-odds. Stretches (0,1) onto the whole real line."""
    return np.log(p / (1.0 - p))

def sigmoid(z):
    """Log-odds -> probability. The inverse of logit (logistic regression's output)."""
    return 1.0 / (1.0 + np.exp(-z))

p = 0.75
z = logit(p)                 # forward:  probability -> log-odds
p_back = sigmoid(z)          # backward: log-odds -> probability

print(f"p             = {p}")
print(f"odds          = p/(1-p) = {p/(1-p):.4f}")
print(f"logit(p)      = ln(odds) = {z:.4f}   (log-odds)")
print(f"sigmoid(z)    = {p_back:.4f}   (round-trips back to p)")
print(f"round-trip exact? {np.isclose(p, p_back)}")

# p and (1-p) give equal-and-opposite log-odds — the log scale's mirror property again
print(f"\nlogit(0.90) = {logit(0.90):+.4f}   logit(0.10) = {logit(0.10):+.4f}")
print(f"mirror image? {np.isclose(logit(0.90), -logit(0.10))}")

p             = 0.75
odds          = p/(1-p) = 3.0000
logit(p)      = ln(odds) = 1.0986   (log-odds)
sigmoid(z)    = 0.7500   (round-trips back to p)
round-trip exact? True

logit(0.90) = +2.1972   logit(0.10) = -2.1972
mirror image? True


## 4. Why it matters — multiplying many probabilities

Multiply a thousand small probabilities and the product **underflows** to `0.0`. Add their
**logs** instead and the score stays in a normal range. This is the product-to-sum law from
section 1 doing real work (naive Bayes, log-likelihood, cross-entropy).

In [5]:
probs = np.full(1000, 0.001)      # 1000 tiny word-probabilities

raw_product = np.prod(probs)      # multiply them all -> underflow
log_sum = np.sum(np.log(probs))   # add their logs -> stable

print(f"raw product of 1000 probs = {raw_product}   (underflowed to 0!)")
print(f"sum of log-probs          = {log_sum:.2f}   (a perfectly ordinary number)")
print(f"recovered via exp?        = {np.exp(log_sum)}   (still 0 when exp'd, but comparable in log space)")
print("\nlesson: stay in log space to compare likelihoods; exponentiate only at the very end")

raw product of 1000 probs = 0.0   (underflowed to 0!)
sum of log-probs          = -6907.76   (a perfectly ordinary number)
recovered via exp?        = 0.0   (still 0 when exp'd, but comparable in log space)

lesson: stay in log space to compare likelihoods; exponentiate only at the very end


## Recap

- **Exponent and logarithm are inverses**; `exp` and `ln` undo each other.
- **`log(a·b) = log a + log b`** — multiplication becomes addition, ratios become differences.
- On the **log scale**, `0.5` and `2` are **equidistant from `1`** — why ratio CIs use log space.
- **`logit`** stretches a probability to a log-odds; **`sigmoid`** squashes it back exactly — the bridge to logistic regression.
- Multiplying many probabilities underflows; **add their logs** to stay stable.

← Back to the **Logarithms, Exponents & the Log Scale** article for the full explanation, diagrams, and interview Q&A.